# Calculate climate driver indices

In [1]:
from dask.distributed import Client,LocalCluster
from dask_jobqueue import PBSCluster

In [55]:
# One node on Gadi has 48 cores - try and use up a full node before going to multiple nodes (jobs)

walltime = '01:00:00'
cores = 24
memory = str(4 * cores) + 'GB'

cluster = PBSCluster(walltime=str(walltime), cores=cores, memory=str(memory), processes=cores,
                     job_extra_directives=['-q normal',
                                           '-P w42',
                                           '-l ncpus='+str(cores),
                                           '-l mem='+str(memory),
                                           '-l storage=gdata/w42+gdata/rt52+gdata/xv83'],
                     local_directory='$TMPDIR',
                     job_directives_skip=["select"])
                     # python=os.environ["DASK_PYTHON"])

In [56]:
cluster.scale(jobs=1)
client = Client(cluster)

In [57]:
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://10.6.121.6:35755,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [5]:
# client.restart()

In [6]:
import xarray as xr
import numpy as np
import pandas as pd

import os

In [ ]:
%cd /g/data/w42/dr6273/work/seasonal_energy/

import functions as fn

In [7]:
%load_ext autoreload
%autoreload 2

In [8]:
years = range(1940, 2024)
data_fp = '/g/data/w42/dr6273/work/data/'

## SST data

In [25]:
sst = xr.open_mfdataset(
    "/g/data/w42/dr6273/work/data/hadisst/sst/HadISST_sst.nc"
)["sst"]

In [27]:
sst = sst.sel(time=slice(str(years[0]), str(years[-1])))

In [31]:
sst_anom = sst.groupby("time.month").apply(lambda x: x - x.mean("time"))

/g/data/w42/dr6273/apps/conda/envs/pangeo/lib/python3.10/site-packages/xarray/core/indexing.py:1374: PerformanceWarning: Slicing with an out-of-order index is generating 84 times more chunks
  return self.array[key]


In [34]:
sst_anom = sst_anom.chunk({"time": -1, "latitude": 250, "longitude": 500})

In [38]:
sst_anom.to_dataset(name="sst_anom").to_zarr(
    "/g/data/w42/dr6273/work/data/hadisst/sst/HadISST_sst_anom_" + str(years[0]) + "-" + str(years[-1]) + ".zarr",
    consolidated=True,
    mode="w"
)

In [39]:
sst_anom = xr.open_zarr(
    "/g/data/w42/dr6273/work/data/hadisst/sst/HadISST_sst_anom_" + str(years[0]) + "-" + str(years[-1]) + ".zarr",
    consolidated=True
).sst_anom

### ENSO and IOD indices

In [50]:
def calc_nino34(da, lon_name="longitude", lat_name="latitude"):
    """
    Compute Nino34 index
    """
    return da.sel({lat_name: slice(5, -5), lon_name: slice(-170, -120)}).mean([lat_name, lon_name])

In [55]:
def calc_dmi(da, lon_name="longitude", lat_name="latitude"):
    """
    Calculate Dipole Mode Index
    """    
    da_W = da.sel({lat_name: slice(10, -10), lon_name: slice(50, 70)}).mean([lat_name, lon_name])
    da_E = da.sel({lat_name: slice(0, -10), lon_name: slice(90, 110)}).mean([lat_name, lon_name])
    
    return (da_W - da_E)

In [57]:
sst_anom = xr.open_zarr(
    "/g/data/w42/dr6273/work/data/hadisst/sst/HadISST_sst_anom_" + str(years[0]) + "-" + str(years[-1]) + ".zarr",
    consolidated=True
)

In [63]:
nino34 = calc_nino34(sst_anom).rename({"sst_anom": "nino34"})

In [69]:
nino34["nino34_detrended"] = fn.detrend_dim(nino34.nino34, "time")

In [73]:
dataset = 'hadisst'
print(sst_fp)

/g/data/w42/dr6273/work/data/hadisst/sst/sst_anom_hadisst_moda_sfc_1940-2023.zarr


In [74]:
nino34_fp = data_fp + dataset + '/climate_modes/'+dataset+'_nino34_'+str(years[0])+'-'+str(years[-1])+'.zarr'
print(nino34_fp)

/g/data/w42/dr6273/work/data/hadisst/climate_modes/hadisst_nino34_1940-2023.zarr


In [75]:
nino34.to_zarr(nino34_fp, mode='w', consolidated=True)

In [70]:
dmi = calc_dmi(sst_anom).rename({"sst_anom": "dmi"})

In [71]:
dmi["dmi_detrended"] = fn.detrend_dim(dmi.dmi, "time")

In [76]:
dmi_fp = data_fp+dataset+'/climate_modes/'+dataset+'_dmi_'+str(years[0])+'-'+str(years[-1])+'.zarr'
print(dmi_fp)

/g/data/w42/dr6273/work/data/hadisst/climate_modes/hadisst_dmi_1940-2023.zarr


In [77]:
dmi.to_zarr(dmi_fp, mode='w', consolidated=True)

## MSLP data

Currently, pre-1959 data is only available hourly on NCI

In [72]:
def open_era_data(root_path, variable, years, preprocess=None, concat_dim='time'):
    """
    Open ERA5 data from NCI
    """
    ds_list = []
    for year in years:
        fp = root_path+variable+'/'+str(year)+'/*.nc'
        ds_list.append(xr.open_mfdataset(fp, preprocess=preprocess))
    return xr.concat(ds_list, dim=concat_dim)

1959 onwards

In [73]:
mslp = open_era_data("/g/data/rt52/era5/single-levels/monthly-averaged/", "msl", range(1959, 2024))

In [74]:
mslp = mslp.chunk({"time": -1, "latitude": 250, "longitude": 500})

In [28]:
mslp.to_zarr(
    "/g/data/w42/dr6273/work/data/era5/mslp/mslp_era5_moda_sfc_1959-2023.zarr",
    mode="w",
    consolidated=True
)

In [75]:
mslp = xr.open_zarr(
    "/g/data/w42/dr6273/work/data/era5/mslp/mslp_era5_moda_sfc_1959-2023.zarr",
    consolidated=True
)

Pre 1959 - intermediate step save each year as monthly

In [ ]:
# def preprocess_mslp(ds):
#     """
#     Resample to monthly averages
#     """
#     return ds.resample(time="1MS").mean()

In [ ]:
# client.restart()

In [58]:
for y in range(1957, 1959):
    ds = xr.open_mfdataset(
        "/g/data/rt52/era5/single-levels/reanalysis/msl/" + str(y) + "/*.nc",
        chunks=({"time": 31})
    )
    ds = ds.resample(time="1MS").mean()
    ds = ds.chunk({"time": -1})
    ds.to_netcdf("/g/data/w42/dr6273/work/data/era5/mslp/yearly/mslp_era5_moda_sfc_" + str(y) + ".nc")

Task exception was never retrieved
future: <Task finished name='Task-204226' coro=<Client._gather.<locals>.wait() done, defined at /g/data/w42/dr6273/apps/conda/envs/pangeo/lib/python3.10/site-packages/distributed/client.py:2122> exception=AllExit()>
Traceback (most recent call last):
  File "/g/data/w42/dr6273/apps/conda/envs/pangeo/lib/python3.10/site-packages/distributed/client.py", line 2131, in wait
    raise AllExit()
distributed.client.AllExit


In [76]:
early_mslp = xr.open_mfdataset(
        "/g/data/w42/dr6273/work/data/era5/mslp/yearly/mslp_era5_moda_sfc_*.nc"
)
early_mslp = early_mslp.chunk({"time": -1, "latitude": 250, "longitude": 500})

In [77]:
mslp = xr.concat([early_mslp, mslp], "time").chunk({"time": -1, "latitude": 250, "longitude": 250})

In [78]:
mslp.to_zarr(
    "/g/data/w42/dr6273/work/data/era5/mslp/mslp_era5_moda_sfc_1940-2023.zarr",
    mode="w",
    consolidated=True
)

#### Anomalies

In [79]:
mslp = xr.open_zarr(
    "/g/data/w42/dr6273/work/data/era5/mslp/mslp_era5_moda_sfc_1940-2023.zarr",
    consolidated=True
)

In [80]:
mslp_anom = mslp.groupby("time.month").apply(lambda x: x - x.mean("time"))

/g/data/w42/dr6273/apps/conda/envs/pangeo/lib/python3.10/site-packages/xarray/core/indexing.py:1374: PerformanceWarning: Slicing with an out-of-order index is generating 84 times more chunks
  return self.array[key]


In [81]:
mslp_anom = mslp_anom.chunk({"time": -1, "latitude": 150, "longitude": 510})

In [83]:
years

range(1940, 2024)

In [84]:
mslp_anom.to_zarr(
    "/g/data/w42/dr6273/work/data/era5/mslp/mslp_anom_era5_moda_sfc_" + str(years[0]) + "-" + str(years[-1]) + ".zarr",
    mode="w",
    consolidated=True
)

### Southern Annular Mode index

In [89]:
def calc_sam(mslp_anoms, lat_name='latitude', lon_name='longitude', time_name='time'):
    """
    Calculate Southern Annular Mode index from MSLP anomalies. The SAM index is
    defined as the difference in MSLP anomalies between 40 and 65 degrees South.
    MSLP anomalies are first normalised by dividing by their standard deviation
    (calculated as a function of calendar month).
    """
    
    mslp_40 = mslp_anoms.interp({lat_name: -40}).mean(lon_name)
    mslp_65 = mslp_anoms.interp({lat_name: -65}).mean(lon_name)
    
    norm_40 = mslp_40.groupby(time_name+'.month').apply(lambda x: x / x.std(time_name))
    norm_65 = mslp_65.groupby(time_name+'.month').apply(lambda x: x / x.std(time_name))
    
    return norm_40 - norm_65

In [90]:
mslp_anom = xr.open_zarr(
    "/g/data/w42/dr6273/work/data/era5/mslp/mslp_anom_era5_moda_sfc_" + str(years[0]) + "-" + str(years[-1]) + ".zarr",
    consolidated=True
)

In [96]:
sam = calc_sam(mslp_anom)

/g/data/w42/dr6273/apps/conda/envs/pangeo/lib/python3.10/site-packages/xarray/core/indexing.py:1374: PerformanceWarning: Slicing with an out-of-order index is generating 84 times more chunks
  return self.array[key]
/g/data/w42/dr6273/apps/conda/envs/pangeo/lib/python3.10/site-packages/xarray/core/indexing.py:1374: PerformanceWarning: Slicing with an out-of-order index is generating 84 times more chunks
  return self.array[key]


In [98]:
sam = sam.rename({'msl': 'sam'})

In [100]:
sam["sam_detrended"] = fn.detrend_dim(sam.sam, 'time')

In [103]:
sam_fp = data_fp + 'era5/climate_modes/era5'+'_sam_'+str(years[0])+'-'+str(years[-1])+'.zarr'
sam.to_zarr(sam_fp, mode='w', consolidated=True)

# Close cluster

In [105]:
client.close()
cluster.close()